# Suite BLW — A store given a store below

A store may be given the store below it, from which it loads what it is asked for, when it is asked ([CAP-283: a store given a store below](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/README.md#CAP-283), [CAP-284: objects loaded on demand](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/README.md#CAP-284)): `Builders.Store(below)`. The store below gives an object's document, by its path (`document(path)`): a one-transaction document of its properties and its entries, each object they link by its path. The store above loads an object the first time its values or entries are read, as its committed data, observed by none; an object an entry links that is not yet loaded is held by its path, the same one each time it is named, and loaded when it is read. Given no store below, a store holds what is written to it, as it always has.

Python and TypeScript run this suite with the same case IDs, titles and asserted values. The case setup is below.

In [ ]:
import { Builders as B, Changes, JSON as Json, Plain, References, Repr } from "@mbse/schemas/Framework";
import { LookupError } from "@mbse/schemas/Framework/Errors";
import { assert, raises } from "./support.js";

/** crm.Person, owned by the singleton book by key, with a name and a home; persons know persons. */
function crm(txn: any): any {
  const S = txn.Schemas;
  return txn.uid("crm/1").mutations([(mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p
    .symbols({ Home: S.OfObject, Holds: S.OfRelation, Knows: S.OfRelation, Book: S.OfObject,
      Person: S.OfObject, Keeps: S.OfRelation, Pet: S.OfObject, Walks: S.OfRelation })
    .spec([
      (_: any) => _.Home.name("crm.Home").properties({ city: S.OfStr, zip: S.OfStr }),
      (_: any) => _.Holds.name("crm.Holds").links([(t: any) => t.name("book"), (t: any) => t.name("contact")])
        .properties({ key: S.OfStr }).uniques([["book", "key"], ["contact"]]),
      (_: any) => _.Knows.name("crm.Knows").links([(t: any) => t.name("knower"), (t: any) => t.name("known")]),
      (_: any) => _.Keeps.name("crm.Keeps").links([(t: any) => t.name("keeper"), (t: any) => t.name("pet")])
        .properties({ name: S.OfStr }).uniques([["keeper", "name"], ["pet"]]),
      (_: any) => _.Walks.name("crm.Walks").links([(t: any) => t.name("walker"), (t: any) => t.name("walked")]),
      (_: any) => _.Pet.name("crm.Pet").ref(true).properties({ kind: S.OfStr }).adjacencies([
        (t: any) => t.name("keeper").relation([_.Keeps]).me("pet").owner(true),
        (t: any) => t.name("walkers").relation([_.Walks]).me("walked")]),
      (_: any) => _.Book.name("crm.Book").ref(true).singleton("crm.TheBook").adjacencies({
        contacts: (t: any) => t.relation([_.Holds]).me("book") }),
      (_: any) => _.Person.name("crm.Person").ref(true).properties({ name: S.OfStr, home: _.Home }).adjacencies([
        (t: any) => t.name("book").relation([_.Holds]).me("contact").owner(true),
        (t: any) => t.name("knows").relation([_.Knows]).me("knower"),
        (t: any) => t.name("known_by").relation([_.Knows]).me("known"),
        (t: any) => t.name("pets").relation([_.Keeps]).me("keeper"),
        (t: any) => t.name("walks").relation([_.Walks]).me("walker"),
      ]),
    ])]))]);
}

const below: any = new B.Store();
below.transaction(crm);
const book = below.singleton("crm.TheBook");
const people: Record<string, any> = {};
below.transaction(() => {
  for (const n of ["ada", "bob", "cy"]) {
    people[n] = below.crm.Person((p: any) => p.book((a: any) => a.insert((e: any) => e.key(n).book(book))).name(n));
  }
  below.crm.Person(people.ada, (p: any) => p.home((h: any) => h.city("Paris").zip("75001"))
    .knows((a: any) => a.insert((e: any) => e.known(people.bob))));
  const rexBelow = below.crm.Pet((p: any) => p.keeper((a: any) => a.insert((e: any) => e.name("rex").keeper(people.ada)))
    .kind("dog"));
  below.crm.Person(people.bob, (p: any) => p.walks((a: any) => a.insert((e: any) => e.walked(rexBelow))));
});
const above: any = new B.Store(below);
above.transaction(crm);

/** How many objects of the store below's the store above has loaded, an int. */
function loaded(store: any = above): bigint {
  return BigInt(store._loaded.size);
}

/** The entry of an adjacency whose key is `key`. */
function keyed(adjacency: any, key: string): any {
  return [...adjacency].find((e: any) => e.key.get() === key);
}

/** Plain data: a path written as an object literal, as `Map`s. */
function plain(literal: unknown): any {
  return Json.loads(JSON.stringify(literal));
}

<a id="BLW-01"></a>
## BLW-01 · Nothing is loaded until it is read; reading an object loads it alone, the objects its entries link each held by its path until it is read

In [ ]:
const ours = above.singleton("crm.TheBook");
const before = loaded();
const keys = [...ours.contacts].map((e: any) => e.key.get());
const afterBook = loaded();
const ada = keyed(ours.contacts, "ada").contact;
const name = ada.name.get();
assert(Json.dumps([before, keys, afterBook, name, loaded()]) === '[0, ["ada", "bob", "cy"], 1, "ada", 2]');

<a id="BLW-02"></a>
## BLW-02 · An object is the same object each time it is named, through any entry; its value objects and entries are loaded with it; an object reached by its reference loads with the owners its path passes through, and no other; one reached through an entry loads alone, its owner not loaded

In [ ]:
const bob = keyed(ours.contacts, "bob").contact;
assert([...ada.knows][0].known === bob && [...ada.book][0].book === ours);
assert([...bob.known_by][0].knower === ada);
assert(Json.dumps([ada.home.city.get(), ada.home.zip.get(), bob.name.get(), ada.as_schema().name]) === '["Paris", "75001", "bob", "crm.Person"]');
const fresh: any = new B.Store(below);
fresh.transaction(crm);
const rexPath = plain({ singleton: "crm.TheBook", next: { entry: { i: "contacts", o: "book", properties: { key: "ada" },
  next: { entry: { i: "pets", o: "keeper", properties: { name: "rex" }, next: { target: {} } } } } } });
const rex: any = References.resolve(fresh, Plain.FromPlain(fresh)(fresh.registered("References.OfObject"), new Map([["path", rexPath]])));
assert(Json.dumps([loaded(fresh), Repr.repr(rex), rex.kind.get(), loaded(fresh)]) === '[3, "<crm.Pet>", "dog", 3]');
const other: any = new B.Store(below);
other.transaction(crm);
const walker = [...other.singleton("crm.TheBook").contacts].find((e: any) => e.key.get() === "bob").contact;
const walked = [...walker.walks][0].walked;
assert(Json.dumps([loaded(other), Repr.repr(walked), walked.kind.get(), loaded(other)]) === '[2, "<crm.Pet>", "dog", 3]');

<a id="BLW-03"></a>
## BLW-03 · The store below gives an object's document by its path, each object it links by its path; none where it holds none, and reading such an object above is refused

In [ ]:
const path = plain({ singleton: "crm.TheBook", next: { entry: { i: "contacts", o: "book", properties: { key: "cy" },
  next: { target: {} } } } });
const document = below.document(path);
const nobody = plain({ singleton: "crm.TheBook", next: { entry: { i: "contacts", o: "book",
  properties: { key: "zed" }, next: { target: {} } } } });
assert(Json.dumps([Json.dumps(document), below.document(nobody)]) === '["{\\"mutations\\": [{\\"upsert\\": {\\"exists\\": [{\\"symbols\\": {\\"0\\": {\\"schema\\": [{\\"$ref\\": {\\"singleton\\": \\".\\", \\"next\\": {\\"entry\\": {\\"i\\": \\"schemas\\", \\"o\\": \\"\\", \\"properties\\": {\\"name\\": \\"crm\\"}, \\"next\\": {\\"entry\\": {\\"i\\": \\"schemas\\", \\"o\\": \\"\\", \\"properties\\": {\\"name\\": \\"Person\\"}, \\"next\\": {\\"target\\": {}}}}}}}}]}}, \\"spec\\": [{\\"0\\": [{\\"as_property\\": {\\"name\\": \\"name\\", \\"mutations\\": [{\\"set\\": {\\"value\\": \\"cy\\"}}]}}, {\\"as_adjacency\\": {\\"name\\": \\"book\\", \\"mutations\\": [{\\"insert\\": {\\"entries\\": [{\\"mutations\\": [{\\"as_link\\": {\\"name\\": \\"book\\", \\"mutations\\": [{\\"set\\": {\\"ref\\": {\\"path\\": {\\"singleton\\": \\"crm.TheBook\\", \\"next\\": {\\"target\\": {}}}}}}]}}, {\\"as_property\\": {\\"name\\": \\"key\\", \\"mutations\\": [{\\"set\\": {\\"value\\": \\"cy\\"}}]}}]}]}}]}}]}]}]}}]}", null]');
raises(LookupError, () => above._identity(nobody).name.get(), "the store below holds no object at this path");

<a id="BLW-04"></a>
## BLW-04 · Loading is no transaction of the store's: its followers are given nothing of it, and a transaction open while an object loads keeps its own changes alone

In [ ]:
const effects: any[] = [];
const end = Changes.observe(above, (effect) => effects.push(effect));
const cy = keyed(ours.contacts, "cy").contact;
let seen: string = "";
above.transaction(() => {
  seen = cy.name.get(); // loaded within the transaction
  above.crm.Person(cy, (p: any) => p.name("Cy"));
});
end();
assert(effects.length === 1);
const [effect] = effects;
assert(Json.dumps([seen, cy.name.get(), Json.dumps(Plain.ToPlain(above).Transaction(effect))]) === '["cy", "Cy", "{\\"mutations\\": [{\\"upsert\\": {\\"exists\\": [{\\"symbols\\": {\\"o0\\": {\\"reference\\": {\\"path\\": {\\"singleton\\": \\"crm.TheBook\\", \\"next\\": {\\"entry\\": {\\"i\\": \\"contacts\\", \\"o\\": \\"book\\", \\"properties\\": {\\"key\\": \\"cy\\"}, \\"next\\": {\\"target\\": {}}}}}}}}, \\"spec\\": [{\\"o0\\": [{\\"as_property\\": {\\"name\\": \\"name\\", \\"mutations\\": [{\\"set\\": {\\"value\\": \\"Cy\\"}}]}}]}]}]}}]}"]');

<a id="BLW-05"></a>
## BLW-05 · An object loaded has its reference as the store below has it; given no store below, a store holds what is written to it

In [ ]:
const alone: any = new B.Store();
alone.transaction(crm);
alone.transaction(() => {
  alone.crm.Person((p: any) => p.book((a: any) => a.insert((e: any) => e.key("dee").book(alone.singleton("crm.TheBook")))));
});

/** An object's reference, as plain data. */
function pathOf(store: any, x: unknown): string {
  return Json.dumps(Plain.ToPlain(store)(store.registered("References.OfObject"), References.of(store, x)));
}

assert(Json.dumps([pathOf(above, cy) === pathOf(below, people.cy),
  [...alone.singleton("crm.TheBook").contacts].map((e: any) => e.key.get()), alone._below]) === '[true, ["dee"], null]');


<a id="BLW-06"></a>
## BLW-06 · An object the store above makes is found again by its path, and one the store below adds after its owner is loaded above is loaded when it is named

In [ ]:
let eve: any = null;
above.transaction(() => {
  eve = above.crm.Person((p: any) => p.book((a: any) => a.insert((e: any) => e.key("eve").book(ours))).name("eve"));
});
const found = References.resolve(above, References.of(above, eve));
below.transaction(() => {
  below.crm.Person((p: any) => p.book((a: any) => a.insert((e: any) => e.key("fay").book(book))).name("fay"));
});
const fayPath = Json.loads('{"singleton": "crm.TheBook", "next": {"entry": {"i": "contacts", "o": "book", "properties": {"key": "fay"}, "next": {"target": {}}}}}');
const fay: any = References.resolve(above, Plain.FromPlain(above)(above.registered("References.OfObject"), Json.loads(`{"path": ${Json.dumps(fayPath)}}`)));
assert(Json.dumps([found === eve, fay.name.get()]) === '[true, "fay"]');